# Random Forest Regression
- ensemble model that uses multiple trees to make more stable predictions

In [4]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder

#import from the data_splitter Python script to load and split the data - prevents having to copy and paste code
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..'))) #used to move to the parent folder
from src.data_splitter import load_data, split

#load dataset and variables
df, X, y = load_data()

#split data by groups
X_train, X_val, X_test, y_train, y_val, y_test = split(df, X, y)

Training: 10661 rows | 70 unique players
Validation: 2169 rows | 15 unique players
Testing: 2170 rows | 15 unique players


In [5]:
#need to use one hot encoding to encode movement_type because it's a string - models can only work with numerical values
categorical = ["movement_type"]
numeric = [col for col in X_train.columns if col not in categorical]

#preprocess data: since it's a tree, don't need to scale numerical features but categorical features need to be converted to numeric using OneHotEncoding
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical)
    ],
    remainder="passthrough" #keeps but doesn't change numerical info
)

#combine preprocessor and random forest regression in a pipeline (does it in sequential steps)
rf_pipeline0 = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", RandomForestRegressor(
            n_estimators=100,
            random_state=42,
            oob_score=True
        ))
    ]
)

rf_pipeline0.fit(X_train, y_train)
print("Random forest pipeline (without tuning) fitted successfully on training data")

Random forest pipeline (without tuning) fitted successfully on training data


In [22]:
#evaluation on random forest regressor using validation data
#predict target values for unseen validation data
y_val_pred = rf_pipeline0.predict(X_val)

#calculate regression metrics: mean absolute error, root mean squared error, R^2
val_mae = mean_absolute_error(y_val, y_val_pred)
val_rmse = root_mean_squared_error(y_val, y_val_pred)
val_r2 = r2_score(y_val, y_val_pred)

#performance summary
print("=== RANDOM FOREST REGRESSOR MODEL 0 METRICS (NO TUNING) (Validation Set) ===")
print(f"MAE  : {val_mae:.4f}")
print(f"RMSE : {val_rmse:.4f}")
print(f"R²   : {val_r2:.4f}")

=== RANDOM FOREST REGRESSOR MODEL 0 METRICS (NO TUNING) (Validation Set) ===
MAE  : 3.4478
RMSE : 4.3417
R²   : 0.6120


# Random Forest Regressor is doing worse than baseline linear regression
- will try tuning the hyperparameters to do better than baseline
- will 1st tune the core parameters: n_estimators, max_features, max_depth, min_samples_split, min_samples_leaf

# Tuning try 1
- n_estimators - number of trees --> n_estimators=150
  - more trees generally improves performance but higher computational cost
- max_features - limits number of features to consider
  - might be helpful since there are so many features, default is sqrt
  - let's try log2 for more randomness

In [24]:
#trying to tune random forest on n_estimators and max_features
rf_pipeline1 = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", RandomForestRegressor(
            n_estimators=150,
            random_state=42,
            oob_score=True,
            max_features="log2"
        ))
    ]
)

rf_pipeline1.fit(X_train, y_train)
print("Random forest pipeline 1 fitted successfully on training data")

Random forest pipeline 1 fitted successfully on training data


In [25]:
#evaluation on random forest regressor 1 using validation data
#predict target values for unseen validation data
y_val_pred = rf_pipeline1.predict(X_val)

#calculate regression metrics: mean absolute error, root mean squared error, R^2
val_mae = mean_absolute_error(y_val, y_val_pred)
val_rmse = root_mean_squared_error(y_val, y_val_pred)
val_r2 = r2_score(y_val, y_val_pred)

#performance summary
print("=== RANDOM FOREST REGRESSOR MODEL 1 METRICS (TUNED ON n_estimators and max_features) (Validation Set) ===")
print(f"MAE  : {val_mae:.4f}")
print(f"RMSE : {val_rmse:.4f}")
print(f"R²   : {val_r2:.4f}")

=== RANDOM FOREST REGRESSOR MODEL 1 METRICS (TUNED ON n_estimators and max_features) (Validation Set) ===
MAE  : 3.6191
RMSE : 4.5793
R²   : 0.5683


# RANDOM FOREST REGRESSOR MODEL 1 performed worse than the baseline Random Forest Regressor
- changing max_features to log2 wasn't helpful
- seems like the model is memorizing the training data (overfitting)

# Tuning on just max_depth to see which max_depth gets the best result
- max_depth - shallow tree underfits, deep tree over fits
  - try decreasing to prevent overfitting

In [26]:
#trying to tune random forest on max_depth
depths = [4, 8, 12, 16, 20, 25, None]

#loop through the different depths to see which one is best
for d in depths:
    rf_pipeline_max_depth = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("regressor", RandomForestRegressor(
                n_estimators=150,
                random_state=42,
                oob_score=True,
                max_depth=d
            ))
        ]
    )

    #train and compare the depths
    rf_pipeline_max_depth.fit(X_train, y_train)
    pred = rf_pipeline_max_depth.predict(X_val)
    mae = mean_absolute_error(y_val, pred)
    rmse = root_mean_squared_error(y_val, pred)
    r2 = r2_score(y_val, pred)
    
    print(f"Max Depth: {str(d):<5}| MAE: {mae:.4f} | RMSE: {rmse:.4f} | R²: {r2:.4f}")

Max Depth: 4    | MAE: 4.1312 | RMSE: 5.1916 | R²: 0.4452
Max Depth: 8    | MAE: 3.5068 | RMSE: 4.4179 | R²: 0.5982
Max Depth: 12   | MAE: 3.4420 | RMSE: 4.3375 | R²: 0.6127
Max Depth: 16   | MAE: 3.4402 | RMSE: 4.3364 | R²: 0.6129
Max Depth: 20   | MAE: 3.4406 | RMSE: 4.3344 | R²: 0.6133
Max Depth: 25   | MAE: 3.4370 | RMSE: 4.3309 | R²: 0.6139
Max Depth: None | MAE: 3.4405 | RMSE: 4.3357 | R²: 0.6130


# want low MAE (less important), low RMSE, and high R²
- MAE, RMSE, R² plateau around 12
- will go with max_depth=12

# Tuning on min_samples_split, use max_depth=12
- min_samples_split - minimum number of samples required to split internal node
- increase so that a leaf node will only be created if it represents a meaningful number of leaf nodes

In [7]:
#trying to tune random forest on min_samples_split
split_samples = [2, 5, 10, 20, 50, 100]

#loop through the different depths to see which one is best
for s in split_samples:
    rf_pipeline_min_samples_split = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("regressor", RandomForestRegressor(
                n_estimators=150,
                random_state=42,
                oob_score=True,
                max_depth=12,
                min_samples_split=s
            ))
        ]
    )

    #train and compare the depths
    rf_pipeline_min_samples_split.fit(X_train, y_train)
    pred = rf_pipeline_min_samples_split.predict(X_val)
    mae = mean_absolute_error(y_val, pred)
    rmse = root_mean_squared_error(y_val, pred)
    r2 = r2_score(y_val, pred)
    
    print(f"Min Samples Split: {str(s):<5}| MAE: {mae:.4f} | RMSE: {rmse:.4f} | R²: {r2:.4f}")

Min Samples Split: 2    | MAE: 3.4420 | RMSE: 4.3375 | R²: 0.6127
Min Samples Split: 5    | MAE: 3.4410 | RMSE: 4.3304 | R²: 0.6140
Min Samples Split: 10   | MAE: 3.4396 | RMSE: 4.3330 | R²: 0.6135
Min Samples Split: 20   | MAE: 3.4408 | RMSE: 4.3351 | R²: 0.6131
Min Samples Split: 50   | MAE: 3.4623 | RMSE: 4.3650 | R²: 0.6078
Min Samples Split: 100  | MAE: 3.5319 | RMSE: 4.4540 | R²: 0.5916


- will go with min_sample_split=5

# max_features and min_samples_leaf should be tested together
- max_features - (horizontal diversity) limits number of features to consider
  - lower for overfitting
  - will try fractions this time
  - 1 means all features
  - want to stop model from relying on the exact same features
- min_samples_leaf - (vertical/depth regularization) min number of data points that live in a final leaf node
  - raise for overfitting
  - overfitting causes it to create tiny leaf nodes
- use max_depth=12, min_sample_split=5

In [11]:
#trying to tune random forest on max_features and min_samples_leaf
feature_options = [.2, .4, .6, .8, 1]
leaf_options = [1, 5, 20, 50]

#loop through the different depths to see which one is best
for f in feature_options:
    for l in leaf_options:
        rf_pipeline_features_leaves = Pipeline(
            steps=[
                ("preprocessor", preprocessor),
                ("regressor", RandomForestRegressor(
                    n_estimators=150,
                    random_state=42,
                    oob_score=True,
                    max_depth=12,
                    min_samples_split=5,
                    max_features=f,
                    min_samples_leaf=l
                ))
            ]
        )

        #train and compare the depths
        rf_pipeline_features_leaves.fit(X_train, y_train)
        pred = rf_pipeline_features_leaves.predict(X_val)
        mae = mean_absolute_error(y_val, pred)
        rmse = root_mean_squared_error(y_val, pred)
        r2 = r2_score(y_val, pred)
    
        print(f"Max Features: {str(f):<5}| Min Samples Leaf: {str(l):<5}| MAE: {mae:.4f} | RMSE: {rmse:.4f} | R²: {r2:.4f}")#trying to tune random forest on max_features and min_samples_leaf
feature_options = [.2, .4, .6, .8, 1]
leaf_options = [1, 5, 20, 50]

#loop through the different depths to see which one is best
for f in feature_options:
    for l in leaf_options:
        rf_pipeline_features_leaves = Pipeline(
            steps=[
                ("preprocessor", preprocessor),
                ("regressor", RandomForestRegressor(
                    n_estimators=150,
                    random_state=42,
                    oob_score=True,
                    max_depth=12,
                    min_samples_split=5,
                    max_features=f,
                    min_samples_leaf=l
                ))
            ]
        )

        #train and compare the depths
        rf_pipeline_features_leaves.fit(X_train, y_train)
        pred = rf_pipeline_features_leaves.predict(X_val)
        mae = mean_absolute_error(y_val, pred)
        rmse = root_mean_squared_error(y_val, pred)
        r2 = r2_score(y_val, pred)
    
        print(f"Max Features: {str(f):<5}| Min Samples Leaf: {str(l):<5}| MAE: {mae:.4f} | RMSE: {rmse:.4f} | R²: {r2:.4f}")

Max Features: 0.2  | Min Samples Leaf: 1    | MAE: 3.6245 | RMSE: 4.5690 | R²: 0.5703
Max Features: 0.2  | Min Samples Leaf: 5    | MAE: 3.6407 | RMSE: 4.6034 | R²: 0.5638
Max Features: 0.2  | Min Samples Leaf: 20   | MAE: 3.7423 | RMSE: 4.7256 | R²: 0.5403
Max Features: 0.2  | Min Samples Leaf: 50   | MAE: 3.8999 | RMSE: 4.9207 | R²: 0.5016
Max Features: 0.4  | Min Samples Leaf: 1    | MAE: 3.4546 | RMSE: 4.3615 | R²: 0.6084
Max Features: 0.4  | Min Samples Leaf: 5    | MAE: 3.4512 | RMSE: 4.3570 | R²: 0.6092
Max Features: 0.4  | Min Samples Leaf: 20   | MAE: 3.5116 | RMSE: 4.4358 | R²: 0.5950
Max Features: 0.4  | Min Samples Leaf: 50   | MAE: 3.6239 | RMSE: 4.5812 | R²: 0.5680
Max Features: 0.6  | Min Samples Leaf: 1    | MAE: 3.4259 | RMSE: 4.3269 | R²: 0.6146
Max Features: 0.6  | Min Samples Leaf: 5    | MAE: 3.4318 | RMSE: 4.3350 | R²: 0.6132
Max Features: 0.6  | Min Samples Leaf: 20   | MAE: 3.4909 | RMSE: 4.4020 | R²: 0.6011
Max Features: 0.6  | Min Samples Leaf: 50   | MAE: 3.5

- best performance when max_features=.8 and min_samples_leaf=5
- try to see if we can do any better by really trying to focus around those values

In [16]:
#trying to tune around max_features=.8 and min_samples_leaf=5 to get the best value
focused_features = [0.8, 0.85, 0.9, 0.95]
focused_leaves = [5, 8, 10, 15, 18]

#loop through the different focused values
for f in focused_features:
    for l in focused_leaves:
        rf_pipeline_focused_features_leaves = Pipeline(
            steps=[
                ("preprocessor", preprocessor),
                ("regressor", RandomForestRegressor(
                    n_estimators=150,
                    random_state=42,
                    oob_score=True,
                    max_depth=12,
                    min_samples_split=5,
                    max_features=f,
                    min_samples_leaf=l,
                ))
            ]
        )

        #train and evaluate each combination inside the loops
        rf_pipeline_focused_features_leaves.fit(X_train, y_train)
        pred = rf_pipeline_focused_features_leaves.predict(X_val)
        
        mae = mean_absolute_error(y_val, pred)
        rmse = root_mean_squared_error(y_val, pred)
        r2 = r2_score(y_val, pred)
    
        print(f"Max Features: {str(f):<5} | Min Samples Leaf: {str(l):<5} | MAE: {mae:.4f} | RMSE: {rmse:.4f} | R²: {r2:.4f}")

Max Features: 0.8   | Min Samples Leaf: 5     | MAE: 3.4222 | RMSE: 4.3235 | R²: 0.6152
Max Features: 0.8   | Min Samples Leaf: 8     | MAE: 3.4373 | RMSE: 4.3355 | R²: 0.6131
Max Features: 0.8   | Min Samples Leaf: 10    | MAE: 3.4473 | RMSE: 4.3489 | R²: 0.6107
Max Features: 0.8   | Min Samples Leaf: 15    | MAE: 3.4690 | RMSE: 4.3762 | R²: 0.6058
Max Features: 0.8   | Min Samples Leaf: 18    | MAE: 3.4773 | RMSE: 4.3880 | R²: 0.6036
Max Features: 0.85  | Min Samples Leaf: 5     | MAE: 3.4275 | RMSE: 4.3262 | R²: 0.6147
Max Features: 0.85  | Min Samples Leaf: 8     | MAE: 3.4466 | RMSE: 4.3501 | R²: 0.6105
Max Features: 0.85  | Min Samples Leaf: 10    | MAE: 3.4511 | RMSE: 4.3514 | R²: 0.6102
Max Features: 0.85  | Min Samples Leaf: 15    | MAE: 3.4775 | RMSE: 4.3832 | R²: 0.6045
Max Features: 0.85  | Min Samples Leaf: 18    | MAE: 3.4851 | RMSE: 4.3973 | R²: 0.6020
Max Features: 0.9   | Min Samples Leaf: 5     | MAE: 3.4414 | RMSE: 4.3374 | R²: 0.6127
Max Features: 0.9   | Min Sample

- confirmation that max_features=.8 and min_samples_leaf=5 is the best for the model

# Final Random Forest Regressor Model
- n_estimators=150
- max_depth=12
- min_samples_split=5
- max_features=.8
- min_samples_leaf=5
- unable to do better than the baseline Linear Regression Model

In [17]:
#final random forest regressor once the parameters have been tuned
rf_final = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", RandomForestRegressor(
            n_estimators=150,
            random_state=42,
            oob_score=True,
            max_depth=12,
            min_samples_split=5,
            max_features=.8,
            min_samples_leaf=5
        ))
    ]
)

rf_final.fit(X_train, y_train)
print(" Final Random forest pipeline fitted successfully on training data")

 Final Random forest pipeline fitted successfully on training data


In [18]:
#prediction on the final random forest model and put baseline results side-by-side in print statement
y_val_pred = rf_final.predict(X_val)

#calculate regression metrics: mean absolute error, root mean squared error, R^2
val_mae = mean_absolute_error(y_val, y_val_pred)
val_rmse = root_mean_squared_error(y_val, y_val_pred)
val_r2 = r2_score(y_val, y_val_pred)

#performance summary
print("=== FINAL RANDOM FOREST REGRESSOR MODEL METRICS | BASELINE METRICS ===")
print(f"MAE  : {val_mae:.4f} | 3.1975")
print(f"RMSE : {val_rmse:.4f} | 4.0316")
print(f"R²   : {val_r2:.4f} | 0.6654")

=== FINAL RANDOM FOREST REGRESSOR MODEL METRICS | BASELINE METRICS ===
MAE  : 3.4222 | 3.1975
RMSE : 4.3235 | 4.0316
R²   : 0.6152 | 0.6654


# Conclusion
- Random Forest Regressor is unable to outperform baseline linear regression
- random forest is making worse large mistakes
- smooth linear regression matches the data better than jagged random forest (makes many decisions)